In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import numpy as np
import os

import matplotlib.pyplot as plt
import torchvision
import torchvision.io as torchio
from torchvision.io import ImageReadMode 
from torchvision.transforms import Compose




In [3]:

class PBRDataset(Dataset):
    def __init__(self, input_data_path, pbr_channels=7, transform=None):
        self.input_data_path = input_data_path
        self.transform = transform
        self.pbr_channels = pbr_channels

    def __len__(self):
        return len([data for data in os.listdir(os.path.join(self.input_data_path, "data"))])

    def __getitem__(self, idx):
        # Load multi-channel PBR maps (albedo, normal, roughness, metallic, etc.)
        pbr_map = torch.load(os.path.join(self.input_data_path, "data", f"data_{str(idx)}"))  # Shape: (H, W, C)
        mask = torch.load(os.path.join(self.input_data_path, "masks", f"mask_{str(idx)}")) # Grayscale

        if self.transform:
            transformed = self.transform(image=pbr_map, mask=mask)
            pbr_map = transformed['image']
            mask = transformed['mask']

        return pbr_map.float(), mask.float()
  

# Step 4: Modified Model for Multi-Mask Output
class MultiMaskUNet(nn.Module):
    def __init__(self, in_channels=8, out_channels=2):
        super().__init__()
        
        self.base_model = smp.Unet(
            encoder_name="resnet50",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,  # Output 2 channels
            activation=None  # We'll handle activation separately
        )
        
        # Add final activation
        self.final_activation = nn.Sigmoid()

    def forward(self, x):
        x = self.base_model(x)
        return self.final_activation(x)

# class EnhancedUNet(nn.Module):
#     def __init__(self, in_channels=8, out_channels=1):
#         super().__init__()

#         self.base_model = smp.Unet(
#             encoder_name="resnet50",
#             encoder_weights="imagenet",
#             in_channels=in_channels,
#             classes=out_channels,
#         )

#         # Add CBAM attention to decoder
#         class CBAMBlock(nn.Module):
#             def __init__(self, channel, reduction=16):
#                 super().__init__()
#                 self.avg_pool = nn.AdaptiveAvgPool2d(1)
#                 self.max_pool = nn.AdaptiveMaxPool2d(1)
#                 self.fc = nn.Sequential(
#                     nn.Linear(channel, channel // reduction),
#                     nn.ReLU(inplace=True),
#                     nn.Linear(channel // reduction, channel),
#                     nn.Sigmoid()
#                 )
#                 self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
#                 self.sigmoid = nn.Sigmoid()

#             def forward(self, x):
#                 # Channel attention
#                 ca_avg = self.avg_pool(x)
#                 ca_max = self.max_pool(x)
#                 ca = self.fc(ca_avg.squeeze(-1).squeeze(-1)) + self.fc(ca_max.squeeze(-1).squeeze(-1))
#                 ca = self.sigmoid(ca.unsqueeze(-1).unsqueeze(-1))

#                 # Spatial attention
#                 sa_avg = torch.mean(x, dim=1, keepdim=True)
#                 sa_max, _ = torch.max(x, dim=1, keepdim=True)
#                 sa = torch.cat([sa_avg, sa_max], dim=1)
#                 sa = self.conv(sa)
#                 sa = self.sigmoid(sa)

#                 return x * ca * sa

#         # Modify decoder blocks
#         for i in range(len(self.base_model.decoder.blocks)):
#             self.base_model.decoder.blocks[i] = nn.Sequential(
#                 self.base_model.decoder.blocks[i],
#                 CBAMBlock(self.base_model.decoder.blocks[i][-1].conv2[0].out_channels)
#            )

#     def forward(self, x):
#         return self.base_model(x)

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IN_CHANNELS = 7  # Adjust based on PBR maps
NUM_CLASSES = 1  # Number of damage types
BATCH_SIZE = 4
LR = 0.0001
EPOCHS = 50

# train_transform = Compose([
#     A.RandomRotate90(),
#     A.Flip(),
#     A.Normalize(mean=[0.5]*IN_CHANNELS, std=[0.5]*IN_CHANNELS),
#     ToTensorV2(),
# ])

# Initialize dataset and dataloader (replace with your paths)
train_dataset = PBRDataset(
    input_data_path=OUTPUT_FOLDER,
    transform=None
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model, loss, and optimizer
model = MultiMaskUNet(in_channels=IN_CHANNELS, out_channels=NUM_CLASSES).to(DEVICE)
criterion = smp.losses.DiceLoss(mode='multiclass')
optimizer = optim.Adam(model.parameters(), lr=LR)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\loren\\Documents\\Lorenzo\\progetti personali\\daniele-panerai\\images-and-masks\\torch-data/data'

In [67]:
def loss_fn(preds, targets):
    bce_loss = nn.BCELoss()(preds, targets)
    dice_loss = smp.losses.DiceLoss(mode='binary')(preds, targets)
    return bce_loss + dice_loss

optimizer = optim.Adam(model.parameters(), lr=LR)

# Step 6: Modified Training Loop
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, masks in train_loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {running_loss/len(train_loader)}")


RuntimeError: expected scalar type Byte but found Float

In [ ]:
# Step 7: Inference Function for Multiple Masks
def predict_multi_masks(model, image_path, threshold=0.5):
    model.eval()
    transform = A.Compose([
        A.Normalize(mean=[0.5]*IN_CHANNELS, std=[0.5]*IN_CHANNELS),
        ToTensorV2(),
    ])
    
    pbr_map = np.load(image_path)
    transformed = transform(image=pbr_map)
    input_tensor = transformed["image"].unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        output = model(input_tensor)
        masks = (output.squeeze().cpu().numpy() > threshold).astype(np.uint8)
    
    return masks[0], masks[1]  # Returns both degradation masks


In [ ]:
# Step 8: Visualization for Multiple Masks
def visualize_multi_masks(image_path):
    # Load original PBR maps (first 3 channels for visualization)
    pbr_map = np.load(image_path)[..., :3]
    pbr_map = (pbr_map * 0.5 + 0.5) * 255  # Denormalize
    
    # Get predictions
    mask1, mask2 = predict_multi_masks(model, image_path)
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(pbr_map.astype(np.uint8))
    axes[0].set_title("PBR Map (RGB)")
    axes[1].imshow(mask1, cmap='gray')
    axes[1].set_title("Degradation Type 1")
    axes[2].imshow(mask2, cmap='gray')
    axes[2].set_title("Degradation Type 2")
    plt.show()

# Example usage
visualize_multi_masks("/path/to/test_pbr_map.npy")